In [2]:
import pandas as pd
import pyodbc

In [3]:
server = "localhost"
database = "churnDB"

connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=" + server + ";"
    "DATABASE=" + database + ";"
    "Trusted_Connection=yes;"
)

conn = pyodbc.connect(connection_string, timeout=10)

print("Connected successfully!")

Connected successfully!


In [4]:
query="SELECT*FROM dbo.vw_ChurnData"
df=pd.read_sql(query,conn)
print("Data loaded successfully!")
print("Shape:",df.shape)

C:\Users\nandha\AppData\Local\Temp\ipykernel_6708\2717041655.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query,conn)


Data loaded successfully!
Shape: (7043, 21)


In [5]:
df.isnull().sum()


customerID             0
gender                 0
SeniorCitizen          0
Partner                0
Dependents             0
tenure                 0
PhoneService           0
MultipleLines        682
InternetService        0
OnlineSecurity      1526
OnlineBackup        1526
DeviceProtection    1526
TechSupport         1526
StreamingTV         1526
StreamingMovies     1526
Contract               0
PaperlessBilling       0
PaymentMethod          0
MonthlyCharges         0
TotalCharges          11
Churn                  0
dtype: int64

In [6]:
df=df.dropna()
print("Missing Values removed successfully!")
print("New shape:",df.shape)

Missing Values removed successfully!
New shape: (4832, 21)


In [7]:
# Find columns containing only Yes/No values
yes_no_columns = []

for col in df.columns:
    values = set(df[col].dropna().unique())
    if values.issubset({"Yes", "No"}):
        yes_no_columns.append(col)

# Convert Yes/No to 1/0
for col in yes_no_columns:
    df[col] = df[col].map({"Yes": 1, "No": 0})
print("Yes/No columns converted:")
print(yes_no_columns)

Yes/No columns converted:
[]


In [8]:
# Find Boolean columns
bool_columns = df.select_dtypes(include=["bool"]).columns.tolist()

# Convert True/False to 1/0
for col in bool_columns:
    df[col] = df[col].astype(int)
print("Boolean columns converted to 1/0:")
print(bool_columns)

Boolean columns converted to 1/0:
['SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']


In [9]:
df.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner               int64
Dependents            int64
tenure                int64
PhoneService          int64
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling      int64
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

In [12]:
# One-Hot Encode categorical columns
categorical_columns = df.select_dtypes(include=["object"]).columns.tolist()
df = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

print("One-Hot Encoding completed successfully!")
print("New shape:", df.shape)

One-Hot Encoding completed successfully!
New shape: (4832, 4854)


In [13]:
print("Data types after encoding:")
print(df.dtypes)

print("\nAny remaining missing values:")
print(df.isnull().sum().sum())

Data types after encoding:
SeniorCitizen                            int64
Partner                                  int64
Dependents                               int64
tenure                                   int64
PhoneService                             int64
                                         ...  
Contract_One year                         bool
Contract_Two year                         bool
PaymentMethod_Credit card (automatic)     bool
PaymentMethod_Electronic check            bool
PaymentMethod_Mailed check                bool
Length: 4854, dtype: object

Any remaining missing values:
0


In [17]:
df.to_csv("churn_cleaned.csv", index=False)
print("Cleaned dataset saved successfully!")
print("Final shape:", df.shape)

Cleaned dataset saved successfully!
Final shape: (4832, 4854)


In [19]:
#Remove customerID one-hot encoded columns
customer_id_columns = [col for col in df.columns if col.startswith("customerID_")]
df = df.drop(columns=customer_id_columns)
print("CustomerID columns removed:", len(customer_id_columns))
print("Corrected shape:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())

CustomerID columns removed: 4831
Corrected shape: (4832, 23)
Remaining missing values: 0


In [20]:
df.to_csv("churn_cleaned.csv", index=False)
print("Final cleaned dataset saved successfully!")
print("Final shape:", df.shape)

Final cleaned dataset saved successfully!
Final shape: (4832, 23)


In [21]:
print("Final data types:")
print(df.dtypes)
print("\nObject columns:")
print(df.select_dtypes(include="object").columns.tolist())
print("\nBoolean columns:")
print(df.select_dtypes(include="bool").columns.tolist())

Final data types:
SeniorCitizen                              int64
Partner                                    int64
Dependents                                 int64
tenure                                     int64
PhoneService                               int64
PaperlessBilling                           int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
gender_Male                                 bool
MultipleLines_True                          bool
InternetService_Fiber optic                 bool
OnlineSecurity_True                         bool
OnlineBackup_True                           bool
DeviceProtection_True                       bool
TechSupport_True                            bool
StreamingTV_True                            bool
StreamingMovies_True                        bool
Contract_One year                           bool
Contract_Two year                           bool
Pa

In [22]:
 #Convert Boolean columns to 0/1
bool_columns = df.select_dtypes(include="bool").columns
df[bool_columns] = df[bool_columns].astype(int)
print("Boolean columns converted to 0/1")
print("Object columns:", df.select_dtypes(include="object").columns.tolist())
print("Boolean columns remaining:", df.select_dtypes(include="bool").columns.tolist())

Boolean columns converted to 0/1
Object columns: []
Boolean columns remaining: []


In [23]:
df.to_csv("cleaned_churn_data.csv", index=False)
print("Task 6 dataset saved successfully!")
print("Final shape:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())

Task 6 dataset saved successfully!
Final shape: (4832, 23)
Remaining missing values: 0
